# Tools


Tools用于拓展LLM的能力，使其能够与外部系统、API或自定义函数交互，从而完成仅靠文本无法实现的任务。（搜索、计算、数据库查询）

* 增强 LLM 的功能 ：让 LLM 突破纯文本生成的限制，执行实际操作（如调用搜索引擎、查询数据库、运行代码等）
* 支持智能决策 ：在Agent 工作流中，LLM 根据用户输入动态选择最合适的 Tool 完成任务。
* 模块化设计 ：每个 Tool 专注一个功能，便于复用和组合（例如：搜索工具 + 计算工具 + 天气查
询工具）

* name ：工具的名称
* description ：工具的功能描述
该工具输入的 JSON模式
要调用的函数
* return_direct ：是否应将工具结果直接返回给用户（仅对Agent相关）
默认是False
 
 


#### 1、使用@tool装饰器定义工具

举例1：

In [ ]:
from docutils.nodes import description
from langchain_core.tools import tool, StructuredTool
from pydantic import BaseModel


@tool
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  #默认是函数的名称
print(f"args = {add_number.args}") #参数列表
print(f"description = {add_number.description}")  #默认是函数的说明信息
print(f"return_direct = {add_number.return_direct}")  #默认值是False


name = add_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = 计算两个整数的和
return_direct = False


举例2：

In [2]:
from langchain_core.tools import tool


@tool(name_or_callable="add_two_number", description="add two numbers", return_direct=True)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  #add_two_number
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")  #add two numbers
print(f"return_direct = {add_number.return_direct}")  #True





name = add_two_number
args = {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
description = add two numbers
return_direct = True


In [3]:
#调用工具
add_number.invoke({"a": 10, "b": 20})


30

举例3：修改args参数的描述

In [4]:
from pydantic import Field
from langchain_core.tools import tool
from pydantic import BaseModel


class FieldInfo(BaseModel):
    a: int = Field(description="第1个整型参数")
    b: int = Field(description="第2个整型参数")


@tool(name_or_callable="add_two_number", description="add two numbers", return_direct=True, args_schema=FieldInfo)
def add_number(a: int, b: int) -> int:
    """计算两个整数的和"""
    return a + b


print(f"name = {add_number.name}")  #add_two_number
print(f"args = {add_number.args}")
print(f"description = {add_number.description}")  #add two numbers
print(f"return_direct = {add_number.return_direct}")  #True

name = add_two_number
args = {'a': {'description': '第1个整型参数', 'title': 'A', 'type': 'integer'}, 'b': {'description': '第2个整型参数', 'title': 'B', 'type': 'integer'}}
description = add two numbers
return_direct = True


#### 2、StructuredTool的from_function()的使用

举例1：

In [5]:
from langchain_core.tools.structured import StructuredTool


# 声明一个函数
def search_google(query: str):
    return "最后查询的结果"


# 定义一个工具
search01 = StructuredTool.from_function(
    func=search_google,
    name="Search",
    description="查询google搜索引擎，并将结果返回"
)

print(f"name = {search01.name}")
print(f"args = {search01.args}")
print(f"description = {search01.description}")
print(f"return_direct = {search01.return_direct}")

name = Search
args = {'query': {'title': 'Query', 'type': 'string'}}
description = 查询google搜索引擎，并将结果返回
return_direct = False


In [6]:
search01.invoke({"query":"中美AI的发展现状"})

'最后查询的结果'

举例2：

In [ ]:
from langchain_core.tools.structured import StructuredTool
from pydantic import BaseModel,Field

# 参数
class FieldInfo(BaseModel):
    query: str = Field(description="要检索的关键词")


# 声明一个函数
def search_google(query: str):
    return "最后查询的结果"


# 定义一个工具
search02 = StructuredTool.from_function(
    func=search_google,
    name="Search",
    description="查询google搜索引擎，并将结果返回",
    return_direct=True,
    args_schema=FieldInfo
)

print(f"name = {search02.name}")
print(f"args = {search02.args}")
print(f"description = {search02.description}")
print(f"return_direct = {search02.return_direct}")

name = Search
args = {'query': {'description': '要检索的关键词', 'title': 'Query', 'type': 'string'}}
description = 查询google搜索引擎，并将结果返回
return_direct = True


#### 大模型工具调用

举例1：

大模型分析需要的工具

In [ ]:
# 1、获取大模型
#导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.tools import StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2、获取工具的列表
tools = [MoveFileTool()]  #获取实例，列表可以很多个工具

# 3、因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数:convert_to_openai_function()
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
messages = [HumanMessage(content="将文件a移动到桌面")]

# 5、调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 76, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-Ccv6jf8tx12Z2sftDXdrGZYlm0oA3', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--c77c0b30-2ee6-43b8-b064-ce4c1c8a60d1-0' usage_metadata={'input_tokens': 76, 'output_tokens': 27, 'total_tokens': 103, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


对比

In [11]:
# 获取消息列表
messages = [HumanMessage(content="查询一下明天北京的天气")]

# 调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

print(response)

content='抱歉，我无法提供实时天气信息。建议您查看天气预报网站或使用天气应用程序获取最新的天气信息。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 76, 'total_tokens': 105, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-Ccv9QrtpBiwnLmv2qiTTBDNl6yEdK', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--0bebd8ad-c4ce-49d8-9e9a-3fa67bc1ef0a-0' usage_metadata={'input_tokens': 76, 'output_tokens': 29, 'total_tokens': 105, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


通过上面两个测试发现，得到的AIMessage的核心属性如下：

1、如果分析出需要调用对应的工具：

content：信息为空。因为大模型要调用工具，所以就不会直接返回信息给用户

additional_kwargs：包含function_call字段，指明具体函数调用的参数和函数名。比如：

additional_kwargs={'function_call': {'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}', 'name': 'move_file'}, 'refusal': None}

2、如果分析出不需要调用对应的工具：

content：信息不为空。

additional_kwargs：不包含function_call字段


举例2：

如何调用具体大模型分析出来的工具

1、大模型与Agent的核心区别：是否涉及到工具的调用

2、针对于大模型：仅能分析出要调用的工具，但是此工具（或函数）不能真正的执行

   针对于Agent:除了分析出要调用的工具之外，还可以执行具体的工具（或函数）

In [19]:
# 1、获取大模型
#导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
chat_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2、获取工具的列表
tools = [MoveFileTool()]

# 3、因为大模型invoke调用时，需要传入函数的列表，所以需要将工具转换为函数:convert_to_openai_function()
functions = [convert_to_openai_function(t) for t in tools]

# 4、获取消息列表
messages = [HumanMessage(content="将当前目录下的文件a.txt移动到C:\\Users\\yadddd\\Desktop")]

# 5、调用大模型（传入消息列表、工具的列表）
response = chat_model.invoke(
    input=messages,
    # tools = tools, #不支持
    functions=functions,
)

# print(response)

步骤1：分析调用哪个工具

In [14]:
import json

if "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具：{tool_name} \n 参数：{tool_args}")

else:
    print(f"模型回复：{response.content}")

调用工具：move_file 
 参数：{'source_path': 'a.txt', 'destination_path': 'C:\\Users\\yadddd\\Desktop\\a.txt'}


步骤2：调用

In [16]:
if "move_file" in response.additional_kwargs["function_call"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args)  #调用工具
    print("工具执行的结果", result)

工具执行的结果 File moved successfully from a.txt to C:\Users\yadddd\Desktop\a.txt.
